<a href="https://colab.research.google.com/github/MohammedAyman07/curriculum-rag-research/blob/main/notebooks/01_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CurriculumRAG — Dataset Preparation

## Research Objective
Compare TF-IDF, Dense Semantic Retrieval, and Hybrid Retrieval
for finding relevant educational passages.

## Dataset Domain
Data Structures and Algorithms

## Dataset Components
1. Educational passages
2. Evaluation questions
3. Relevance labels
4. Development and test splits

In [ ]:
# CurriculumRAG - Environment Setup

import os
import sys
import json
import pandas as pd
from pathlib import Path

# Project directories
PROJECT_DIR = Path("/content/curriculum-rag-research")
DATA_DIR = PROJECT_DIR / "data"

# Create directories
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("CurriculumRAG Environment Ready")
print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)

CurriculumRAG Environment Ready
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Pandas version: 2.2.3
Project directory: /content/curriculum-rag-research
Data directory: /content/curriculum-rag-research/data


In [ ]:
# Define dataset schemas

PASSAGE_COLUMNS = [
    "doc_id",
    "topic",
    "text",
    "source"
]

QUERY_COLUMNS = [
    "query_id",
    "question",
    "relevant_doc_ids",
    "split"
]

# Create empty dataset templates
passages_df = pd.DataFrame(columns=PASSAGE_COLUMNS)
queries_df = pd.DataFrame(columns=QUERY_COLUMNS)

print("Passage dataset columns:", list(passages_df.columns))
print("Query dataset columns:", list(queries_df.columns))

Passage dataset columns: ['doc_id', 'topic', 'text', 'source']
Query dataset columns: ['query_id', 'question', 'relevant_doc_ids', 'split']


In [ ]:
def validate_dataset(passages, queries):

    required_passage_columns = PASSAGE_COLUMNS
    required_query_columns = QUERY_COLUMNS

    # Check columns
    assert list(passages.columns) == required_passage_columns
    assert list(queries.columns) == required_query_columns

    # Check duplicate IDs
    assert passages["doc_id"].is_unique, "Duplicate passage IDs found"
    assert queries["query_id"].is_unique, "Duplicate query IDs found"

    # Check empty values
    assert not passages.isnull().any().any(), "Missing passage values"
    assert not queries.isnull().any().any(), "Missing query values"

    # Check relevance IDs
    available_ids = set(passages["doc_id"])

    for doc_ids in queries["relevant_doc_ids"]:
        relevant_ids = set(doc_ids.split("|"))
        assert relevant_ids.issubset(available_ids), (
            f"Unknown passage ID: {relevant_ids - available_ids}"
        )

    # Check split names
    assert set(queries["split"]).issubset({"dev", "test"})

    print("Dataset validation passed!")

In [ ]:

# CurriculumRAG - Create the educational passage dataset

passage_records = [
    {
        "doc_id": "P001",
        "topic": "Time Complexity",
        "text": "Time complexity describes how an algorithm's running time changes as the input size increases. Big-O notation represents an upper bound on the growth rate, helping compare algorithms independently of computer hardware. Common complexities include O(1), O(log n), O(n), O(n log n), O(n^2), and O(2^n)."
    },
    {
        "doc_id": "P002",
        "topic": "Space Complexity",
        "text": "Space complexity measures the memory an algorithm requires relative to its input size. It includes the memory used by variables, data structures, and auxiliary storage. An algorithm with O(1) auxiliary space uses constant extra memory, whereas one using an additional array proportional to input size may require O(n) space."
    },
    {
        "doc_id": "P003",
        "topic": "Abstract Data Types",
        "text": "An Abstract Data Type (ADT) defines a data structure through its permitted operations and expected behaviour rather than its internal implementation. A stack ADT, for example, supports push, pop, and peek operations. It can be implemented using an array or a linked list without changing the ADT's logical behaviour."
    },
    {
        "doc_id": "P004",
        "topic": "Arrays",
        "text": "An array stores elements in contiguous memory locations and allows direct access using an index. Accessing an element by its index typically takes O(1) time. Inserting or deleting an element in the middle may require shifting other elements, making these operations O(n) in a conventional array."
    },
    {
        "doc_id": "P005",
        "topic": "Singly Linked Lists",
        "text": "A singly linked list consists of nodes where each node stores data and a reference to the next node. The list begins at a head pointer, and the final node points to null. Insertion at the beginning takes O(1) time, but accessing an arbitrary position requires sequential traversal."
    },
    {
        "doc_id": "P006",
        "topic": "Doubly Linked Lists",
        "text": "A doubly linked list contains nodes with data, a pointer to the next node, and a pointer to the previous node. This structure supports traversal in both forward and backward directions. It makes deletion convenient when a reference to the target node is available, but requires extra memory for the additional pointer."
    },
    {
        "doc_id": "P007",
        "topic": "Circular Linked Lists",
        "text": "In a circular linked list, the last node points back to the first node instead of pointing to null. This creates a cycle that allows traversal to continue from the end to the beginning. Circular linked lists are useful in round-robin scheduling and applications that repeatedly process elements in sequence."
    },
    {
        "doc_id": "P008",
        "topic": "Stack Data Structure",
        "text": "A stack is a linear data structure that follows the Last In, First Out (LIFO) principle. The most recently inserted element is the first one removed. Its main operations are push for insertion, pop for removal, and peek for viewing the top element. These operations normally take O(1) time."
    },
    {
        "doc_id": "P009",
        "topic": "Stack Applications",
        "text": "Stacks are used in function-call management, expression evaluation, expression conversion, undo operations, and backtracking. During recursion, a program uses a call stack to store information about active function calls. Parentheses matching is another application, where opening symbols are pushed and checked against closing symbols."
    },
    {
        "doc_id": "P010",
        "topic": "Queue Data Structure",
        "text": "A queue is a linear data structure that follows the First In, First Out (FIFO) principle. The first element inserted is the first one removed. Enqueue inserts an element at the rear, while dequeue removes an element from the front. Queues are commonly used in scheduling and buffering."
    },
    {
        "doc_id": "P011",
        "topic": "Circular Queue",
        "text": "A circular queue treats the storage array as a circular structure, allowing the rear position to wrap around to the beginning when space becomes available. This avoids the unused spaces that can occur in a simple array-based queue after repeated deletions. Front and rear indices track the queue positions."
    },
    {
        "doc_id": "P012",
        "topic": "Deque",
        "text": "A double-ended queue, or deque, allows insertion and deletion at both the front and rear. It combines characteristics of stacks and queues. Depending on the implementation, a deque can support operations at either end efficiently and can be used in task scheduling, sliding-window algorithms, and palindrome checking."
    },
    {
        "doc_id": "P013",
        "topic": "Binary Trees",
        "text": "A binary tree is a hierarchical data structure in which each node has at most two children, commonly called the left child and right child. The topmost node is the root, nodes without children are leaves, and edges connect parent and child nodes. Binary trees support several traversal methods."
    },
    {
        "doc_id": "P014",
        "topic": "Binary Search Trees",
        "text": "A Binary Search Tree (BST) is a binary tree that maintains an ordering property. Values in the left subtree of a node are smaller than the node's value, while values in the right subtree are larger under the usual distinct-key convention. Search, insertion, and deletion take O(h), where h is tree height."
    },
    {
        "doc_id": "P015",
        "topic": "AVL Trees",
        "text": "An AVL tree is a self-balancing binary search tree. For every node, the height difference between its left and right subtrees must be at most one. Rotations restore balance after insertion or deletion. Maintaining a balanced height ensures search, insertion, and deletion take O(log n) time."
    },
    {
        "doc_id": "P016",
        "topic": "Tree Traversal",
        "text": "Tree traversal is the process of visiting every node in a tree according to a defined order. In preorder traversal, the root is visited before its subtrees. In inorder traversal, the left subtree is visited before the root and right subtree. Postorder visits the root after both subtrees."
    },
    {
        "doc_id": "P017",
        "topic": "Graph Data Structure",
        "text": "A graph consists of vertices and edges connecting pairs of vertices. Graphs may be directed or undirected, weighted or unweighted, and connected or disconnected. They model networks such as roads, computer connections, and social relationships. Common representations include adjacency matrices and adjacency lists."
    },
    {
        "doc_id": "P018",
        "topic": "Adjacency Matrix",
        "text": "An adjacency matrix represents a graph using a two-dimensional matrix. Each row and column corresponds to a vertex, and a matrix entry indicates whether an edge connects the corresponding vertices or stores its weight. It provides constant-time edge lookup but requires O(V^2) storage for V vertices."
    },
    {
        "doc_id": "P019",
        "topic": "Breadth First Search",
        "text": "Breadth First Search (BFS) explores a graph level by level, visiting neighbouring vertices before moving to vertices farther away. It uses a queue to manage the exploration order and a visited structure to prevent repeated processing. For an adjacency-list representation, its time complexity is O(V + E)."
    },
    {
        "doc_id": "P020",
        "topic": "Depth First Search",
        "text": "Depth First Search (DFS) explores as far as possible along one path before backtracking. It can be implemented using recursion or an explicit stack. DFS is useful for connected-component detection, cycle detection, and graph traversal. With adjacency lists, its time complexity is O(V + E)."
    },
    {
        "doc_id": "P021",
        "topic": "Minimum Spanning Tree",
        "text": "A minimum spanning tree of a connected, weighted, undirected graph is a set of edges that connects all vertices without creating cycles and has the minimum possible total edge weight. A spanning tree with V vertices contains exactly V minus one edges. Prim's and Kruskal's algorithms construct minimum spanning trees."
    },
    {
        "doc_id": "P022",
        "topic": "Prim Algorithm",
        "text": "Prim's algorithm constructs a minimum spanning tree by starting from a selected vertex and repeatedly adding the minimum-weight edge that connects a vertex already in the tree to a vertex outside it. The process continues until all vertices are included. Priority queues can improve its efficiency."
    },
    {
        "doc_id": "P023",
        "topic": "Kruskal Algorithm",
        "text": "Kruskal's algorithm finds a minimum spanning tree by sorting graph edges in non-decreasing order of weight. It considers edges one at a time and adds an edge only when it does not create a cycle. A disjoint-set data structure is commonly used to detect whether two vertices already belong to the same component."
    },
    {
        "doc_id": "P024",
        "topic": "Trie Data Structure",
        "text": "A trie is a tree-based data structure used to store strings by sharing common prefixes. Each path represents a sequence of characters, and nodes can indicate the end of stored words. Tries support prefix searches and dictionary operations and are useful in autocomplete and word-search applications."
    },
    {
        "doc_id": "P025",
        "topic": "Bubble Sort",
        "text": "Bubble Sort repeatedly compares adjacent elements and swaps them when they are in the wrong order. After each complete pass, a largest unsorted element moves to its correct position at the end. Its average and worst-case time complexity is O(n^2), while an optimized version can finish in O(n) when already sorted."
    },
    {
        "doc_id": "P026",
        "topic": "Selection Sort",
        "text": "Selection Sort divides an array into sorted and unsorted portions. During each pass, it finds the minimum element from the unsorted portion and swaps that element into the next sorted position. It performs O(n^2) comparisons regardless of the initial arrangement and typically uses O(1) extra space."
    },
    {
        "doc_id": "P027",
        "topic": "Insertion Sort",
        "text": "Insertion Sort builds a sorted section by taking one element at a time and inserting it into its correct position among the previously processed elements. It is efficient for small or nearly sorted arrays. Its average and worst-case time complexity is O(n^2), while its best case is O(n)."
    },
    {
        "doc_id": "P028",
        "topic": "Merge Sort",
        "text": "Merge Sort is a divide-and-conquer sorting algorithm. It divides an array into smaller parts, recursively sorts those parts, and merges the sorted results. Its time complexity is O(n log n) in the best, average, and worst cases. Standard array implementations require O(n) auxiliary memory."
    },
    {
        "doc_id": "P029",
        "topic": "Quick Sort",
        "text": "Quick Sort selects a pivot element and partitions the array so that elements are arranged on either side according to their relationship with the pivot. It then recursively sorts the partitions. Its average time complexity is O(n log n), although poor pivot choices can cause O(n^2) worst-case time."
    },
    {
        "doc_id": "P030",
        "topic": "Heap Sort",
        "text": "Heap Sort uses a binary heap to organise elements before repeatedly removing the highest-priority element and placing it in its final sorted position. It typically builds a max-heap for ascending order. Its best, average, and worst-case time complexities are O(n log n), with O(1) auxiliary space in an in-place implementation."
    },
    {
        "doc_id": "P031",
        "topic": "Stable Sorting",
        "text": "A sorting algorithm is stable when records with equal sort keys retain their original relative order after sorting. Stability is important when records contain multiple fields and one sorting operation should preserve an earlier ordering. Merge Sort and Insertion Sort are commonly implemented as stable algorithms, while Selection Sort and standard Quick Sort are generally unstable."
    },
    {
        "doc_id": "P032",
        "topic": "Linear Search",
        "text": "Linear Search checks elements sequentially until the target is found or all elements have been examined. It works on both sorted and unsorted collections and requires no preprocessing. Its best-case time complexity is O(1), and its average and worst-case time complexity is O(n)."
    },
    {
        "doc_id": "P033",
        "topic": "Binary Search",
        "text": "Binary Search finds a target in a sorted collection by repeatedly comparing it with the middle element. If the target is smaller, the search continues in the left half; otherwise, it continues in the right half. Each step approximately halves the remaining search space, giving O(log n) time complexity."
    },
    {
        "doc_id": "P034",
        "topic": "Hashing",
        "text": "Hashing maps a key to an index in a hash table using a hash function. The resulting index identifies where the associated value should be stored or searched for. A well-designed hash function distributes keys across the table to reduce collisions and support efficient average-case insertion and lookup."
    },
    {
        "doc_id": "P035",
        "topic": "Hash Collisions",
        "text": "A hash collision occurs when two different keys produce the same hash-table index. Collision handling is necessary because multiple keys may map to the same location. Common approaches include separate chaining and open addressing. The load factor and hash function influence how frequently collisions occur."
    },
    {
        "doc_id": "P036",
        "topic": "Separate Chaining",
        "text": "Separate chaining handles hash collisions by storing multiple entries at the same table index in a linked collection, often a linked list. When a collision occurs, the new entry is added to that index's chain. Search and insertion performance depends on the chain length and overall load factor."
    },
    {
        "doc_id": "P037",
        "topic": "Linear Probing",
        "text": "Linear probing is an open-addressing collision-resolution technique. When a hashed position is occupied, the algorithm checks subsequent table positions until it finds an available slot. The probing sequence advances one position at a time, wrapping around at the end. It can suffer from primary clustering as occupied runs grow."
    },
    {
        "doc_id": "P038",
        "topic": "Quadratic Probing",
        "text": "Quadratic probing resolves collisions in an open-addressing hash table by checking positions at increasing quadratic offsets from the original hash index. This reduces primary clustering compared with linear probing. However, the choice of table size and probing constants affects whether the method can reach enough available positions."
    },
    {
        "doc_id": "P039",
        "topic": "Double Hashing",
        "text": "Double hashing uses a second hash function to determine the step size used when resolving a collision. The next candidate positions are generated using the initial hash value and this step size. It generally reduces clustering compared with linear and quadratic probing when the hash functions and table size are selected appropriately."
    },
    {
        "doc_id": "P040",
        "topic": "Rehashing",
        "text": "Rehashing rebuilds a hash table using a new table size and usually reinserts all existing keys according to the new table's indexing rules. It is commonly performed when the load factor becomes too high and collisions increase. Rehashing takes additional time during resizing but can improve future lookup and insertion performance."
    }
]

# Convert records into a DataFrame
passages_df = pd.DataFrame(passage_records, columns=PASSAGE_COLUMNS)

# Check the generated dataset
assert len(passages_df) == 40, "Expected exactly 40 passages"
assert passages_df["doc_id"].is_unique, "Duplicate passage IDs found"
assert passages_df["text"].str.len().min() > 0, "Empty passage found"

# Save the dataset
passages_path = DATA_DIR / "passages.csv"
passages_df.to_csv(passages_path, index=False)

print("Passage dataset created successfully!")
print("Total passages:", len(passages_df))
print("Unique passage IDs:", passages_df["doc_id"].nunique())
print("Saved to:", passages_path)

display(passages_df.head(10))


Passage dataset created successfully!
Total passages: 40
Unique passage IDs: 40
Saved to: /content/curriculum-rag-research/data/passages.csv


,doc_id,topic,text,source
0,P001,Time Complexity,Time complexity describes how an algorithm's r...,NaN
1,P002,Space Complexity,Space complexity measures the memory an algori...,NaN
2,P003,Abstract Data Types,An Abstract Data Type (ADT) defines a data str...,NaN
3,P004,Arrays,An array stores elements in contiguous memory ...,NaN
4,P005,Singly Linked Lists,A singly linked list consists of nodes where e...,NaN
5,P006,Doubly Linked Lists,"A doubly linked list contains nodes with data,...",NaN
6,P007,Circular Linked Lists,"In a circular linked list, the last node point...",NaN
7,P008,Stack Data Structure,A stack is a linear data structure that follow...,NaN
8,P009,Stack Applications,"Stacks are used in function-call management, e...",NaN
9,P010,Queue Data Structure,A queue is a linear data structure that follow...,NaN


In [ ]:

# CurriculumRAG - Create evaluation questions

# Fix the source field in the passage dataset
passages_df["source"] = (
    "CurriculumRAG original pilot content "
    "(AI-assisted draft; pending manual review)"
)

# 60 questions with manually assigned initial relevance labels
# Format: query_id, question, relevant_doc_ids, split

query_records = [
    # ---------------- DEVELOPMENT SET: 15 ----------------
    ("Q001", "How does an algorithm's running time change as input size increases?", "P001", "dev"),
    ("Q002", "What does auxiliary space complexity tell us about an algorithm?", "P002", "dev"),
    ("Q003", "Can a stack be implemented in different ways while keeping the same operations?", "P003", "dev"),
    ("Q004", "Why can an array element be accessed directly using its index?", "P004", "dev"),
    ("Q005", "How are nodes connected in a singly linked list?", "P005", "dev"),
    ("Q006", "Which linked list supports traversal in both directions?", "P006", "dev"),
    ("Q007", "How does the last node connect in a circular linked list?", "P007", "dev"),
    ("Q008", "Which principle explains why the latest stack element is removed first?", "P008", "dev"),
    ("Q009", "How does a program use a stack to manage recursive function calls?", "P009", "dev"),
    ("Q010", "Where does a queue insert new elements and remove existing elements?", "P010", "dev"),
    ("Q011", "How does a circular queue reuse array positions after deletions?", "P011", "dev"),
    ("Q012", "What makes a deque different from an ordinary queue?", "P012", "dev"),
    ("Q013", "How many children can a node have in a binary tree?", "P013", "dev"),
    ("Q014", "What ordering condition must a binary search tree maintain?", "P014", "dev"),
    ("Q015", "What balance condition does an AVL tree maintain at each node?", "P015", "dev"),

    # ---------------- TEST SET: 45 ----------------
    ("Q016", "In which tree traversal is the root processed between its left and right subtrees?", "P016", "test"),
    ("Q017", "What is the difference between preorder and postorder traversal?", "P016", "test"),
    ("Q018", "What are the main components of a graph, and how can graphs be classified?", "P017", "test"),
    ("Q019", "How much memory does an adjacency matrix need for a graph with V vertices?", "P018", "test"),
    ("Q020", "Which graph traversal visits nearby vertices before exploring more distant ones?", "P019", "test"),
    ("Q021", "What is the time complexity of BFS when a graph uses an adjacency list?", "P019", "test"),
    ("Q022", "How does DFS explore a graph and then return to earlier vertices?", "P020", "test"),
    ("Q023", "Which data structures can be used to implement depth-first search?", "P020", "test"),
    ("Q024", "What properties must a minimum spanning tree satisfy?", "P021", "test"),
    ("Q025", "How does Prim's algorithm decide which edge to add to its growing tree?", "P022", "test"),
    ("Q026", "How does Kruskal's algorithm avoid adding an edge that creates a cycle?", "P023", "test"),
    ("Q027", "Which tree structure stores words by sharing prefixes and supports autocomplete?", "P024", "test"),
    ("Q028", "How does Bubble Sort move large elements toward the end of an array?", "P025", "test"),
    ("Q029", "What element does Selection Sort choose during each pass?", "P026", "test"),
    ("Q030", "Why is Insertion Sort useful when an array is already nearly sorted?", "P027", "test"),
    ("Q031", "How does Merge Sort use the divide-and-conquer approach?", "P028", "test"),
    ("Q032", "What role does a pivot play in Quick Sort, and what causes its worst case?", "P029", "test"),
    ("Q033", "What are the time and auxiliary-space complexities of in-place Heap Sort?", "P030", "test"),
    ("Q034", "What happens to equal-key records when a stable sorting algorithm is used?", "P031", "test"),
    ("Q035", "Which sorting algorithms are commonly stable, and which are generally unstable?", "P031", "test"),
    ("Q036", "Can linear search find a value in an unsorted collection?", "P032", "test"),
    ("Q037", "What is the worst-case time complexity of sequential search?", "P032", "test"),
    ("Q038", "Why must a collection be sorted before applying ordinary binary search?", "P033", "test"),
    ("Q039", "Why does binary search take logarithmic time?", "P033", "test"),
    ("Q040", "How does a hash function determine where a key is stored in a hash table?", "P034", "test"),
    ("Q041", "What happens when two different keys produce the same hash-table index?", "P035", "test"),
    ("Q042", "How does separate chaining store entries that have the same hash index?", "P036", "test"),
    ("Q043", "How does linear probing find an empty position after a collision?", "P037", "test"),
    ("Q044", "How are new positions selected in quadratic probing?", "P038", "test"),
    ("Q045", "How does a second hash function help resolve collisions?", "P039", "test"),
    ("Q046", "Why might a hash table need to be rebuilt with a larger capacity?", "P040", "test"),
    ("Q047", "Which structure provides direct indexed access, while another requires node-by-node traversal?", "P004|P005", "test"),
    ("Q048", "How do stack and queue removal orders differ?", "P008|P010", "test"),
    ("Q049", "How do breadth-first search and depth-first search differ in exploration order?", "P019|P020", "test"),
    ("Q050", "How do Prim's and Kruskal's algorithms construct a minimum spanning tree differently?", "P022|P023", "test"),
    ("Q051", "How do Merge Sort and Quick Sort differ in their worst-case time complexity?", "P028|P029", "test"),
    ("Q052", "How do linear search and binary search differ in their requirements and time complexity?", "P032|P033", "test"),
    ("Q053", "How do collisions and a high load factor affect a hash table, and why is rehashing useful?", "P035|P040", "test"),
    ("Q054", "How does a doubly linked list improve traversal compared with a singly linked list?", "P005|P006", "test"),
    ("Q055", "How does AVL balancing improve on the height behaviour of an ordinary binary search tree?", "P014|P015", "test"),
    ("Q056", "How does an adjacency matrix represent connections between graph vertices?", "P017|P018", "test"),
    ("Q057", "Which sorting method uses a heap, and which one repeatedly merges sorted subarrays?", "P028|P030", "test"),
    ("Q058", "How many edges are present in a spanning tree containing V vertices?", "P021", "test"),
    ("Q059", "Which queue design allows the array's rear index to wrap around after reaching its end?", "P011", "test"),
    ("Q060", "What is the difference between the structure of a binary tree and the order used to visit its nodes?", "P013|P016", "test"),
]

# Convert records into a DataFrame
queries_df = pd.DataFrame(
    query_records,
    columns=QUERY_COLUMNS
)

# ---------------- VALIDATION ----------------

# Confirm number of questions
assert len(queries_df) == 60, "Expected exactly 60 questions"

# Confirm split sizes
assert (queries_df["split"] == "dev").sum() == 15
assert (queries_df["split"] == "test").sum() == 45

# Confirm unique query IDs
assert queries_df["query_id"].is_unique

# Validate passage and query data
validate_dataset(passages_df, queries_df)

# Save both datasets
passages_df.to_csv(DATA_DIR / "passages.csv", index=False)
queries_df.to_csv(DATA_DIR / "queries.csv", index=False)

print("\nEvaluation dataset created successfully!")
print("Total questions:", len(queries_df))
print("Development questions:", (queries_df["split"] == "dev").sum())
print("Test questions:", (queries_df["split"] == "test").sum())
print("Passage records:", len(passages_df))
print("\nFiles saved:")
print(DATA_DIR / "passages.csv")
print(DATA_DIR / "queries.csv")

display(queries_df.head(10))
display(queries_df.tail(5))


Dataset validation passed!

Evaluation dataset created successfully!
Total questions: 60
Development questions: 15
Test questions: 45
Passage records: 40

Files saved:
/content/curriculum-rag-research/data/passages.csv
/content/curriculum-rag-research/data/queries.csv


,query_id,question,relevant_doc_ids,split
0,Q001,How does an algorithm's running time change as...,P001,dev
1,Q002,What does auxiliary space complexity tell us a...,P002,dev
2,Q003,Can a stack be implemented in different ways w...,P003,dev
3,Q004,Why can an array element be accessed directly ...,P004,dev
4,Q005,How are nodes connected in a singly linked list?,P005,dev
5,Q006,Which linked list supports traversal in both d...,P006,dev
6,Q007,How does the last node connect in a circular l...,P007,dev
7,Q008,Which principle explains why the latest stack ...,P008,dev
8,Q009,How does a program use a stack to manage recur...,P009,dev
9,Q010,Where does a queue insert new elements and rem...,P010,dev


,query_id,question,relevant_doc_ids,split
55,Q056,How does an adjacency matrix represent connect...,P017|P018,test
56,Q057,"Which sorting method uses a heap, and which on...",P028|P030,test
57,Q058,How many edges are present in a spanning tree ...,P021,test
58,Q059,Which queue design allows the array's rear ind...,P011,test
59,Q060,What is the difference between the structure o...,P013|P016,test


In [ ]:

# CurriculumRAG - Improve development/test distribution

development_ids = {
    "Q001",  # Time complexity
    "Q004",  # Arrays
    "Q006",  # Doubly linked lists
    "Q008",  # Stacks
    "Q010",  # Queues
    "Q013",  # Binary trees
    "Q015",  # AVL trees
    "Q018",  # Graphs
    "Q020",  # BFS
    "Q022",  # DFS
    "Q025",  # Prim's algorithm
    "Q027",  # Tries
    "Q031",  # Merge Sort
    "Q038",  # Binary Search
    "Q040"   # Hashing
}

# Assign the new split
queries_df["split"] = queries_df["query_id"].apply(
    lambda qid: "dev" if qid in development_ids else "test"
)

# Validate updated dataset
validate_dataset(passages_df, queries_df)

# Check split counts
print("\nUpdated dataset distribution")
print(queries_df["split"].value_counts())

# Save updated datasets
passages_df.to_csv(DATA_DIR / "passages.csv", index=False)
queries_df.to_csv(DATA_DIR / "queries.csv", index=False)

print("\nUpdated CSV files saved successfully!")


Dataset validation passed!

Updated dataset distribution
split
test    45
dev     15
Name: count, dtype: int64

Updated CSV files saved successfully!


In [ ]:

# CurriculumRAG - Relevance Label Audit

# Create a lookup from passage ID to passage text
passage_lookup = passages_df.set_index("doc_id")["text"].to_dict()

# Build a review table
audit_records = []

for _, query in queries_df.iterrows():

    relevant_ids = query["relevant_doc_ids"].split("|")

    relevant_texts = []

    for doc_id in relevant_ids:
        relevant_texts.append(
            f"{doc_id}: {passage_lookup[doc_id]}"
        )

    audit_records.append({
        "query_id": query["query_id"],
        "question": query["question"],
        "relevant_doc_ids": query["relevant_doc_ids"],
        "relevant_passage_text": "\n\n".join(relevant_texts),
        "split": query["split"],
        "review_status": "PENDING"
    })

audit_df = pd.DataFrame(audit_records)

# Check coverage of all passages
labelled_ids = set()

for ids in queries_df["relevant_doc_ids"]:
    labelled_ids.update(ids.split("|"))

all_passage_ids = set(passages_df["doc_id"])

uncovered_ids = sorted(all_passage_ids - labelled_ids)

print("Total questions:", len(audit_df))
print("Total unique passages labelled:", len(labelled_ids))
print("Passages without a relevance label:", uncovered_ids)

# Display questions and their supporting passages
display(audit_df[
    ["query_id", "question", "relevant_doc_ids", "split"]
])

# Export the detailed audit sheet
audit_path = DATA_DIR / "relevance_audit.csv"
audit_df.to_csv(audit_path, index=False)

print("\nRelevance audit sheet saved:", audit_path)


Total questions: 60
Total unique passages labelled: 40
Passages without a relevance label: []


,query_id,question,relevant_doc_ids,split
0,Q001,How does an algorithm's running time change as...,P001,dev
1,Q002,What does auxiliary space complexity tell us a...,P002,test
2,Q003,Can a stack be implemented in different ways w...,P003,test
3,Q004,Why can an array element be accessed directly ...,P004,dev
4,Q005,How are nodes connected in a singly linked list?,P005,test
5,Q006,Which linked list supports traversal in both d...,P006,dev
6,Q007,How does the last node connect in a circular l...,P007,test
7,Q008,Which principle explains why the latest stack ...,P008,dev
8,Q009,How does a program use a stack to manage recur...,P009,test
9,Q010,Where does a queue insert new elements and rem...,P010,dev



Relevance audit sheet saved: /content/curriculum-rag-research/data/relevance_audit.csv


In [ ]:

# CurriculumRAG - Export research datasets

from google.colab import files
from pathlib import Path
import zipfile

required_files = [
    "passages.csv",
    "queries.csv",
    "relevance_audit.csv"
]

# Check that all files exist
for filename in required_files:
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"{filename} is missing. Run the dataset or audit cell first."
        )

# Create a ZIP archive
zip_path = Path("/content/curriculumrag_dataset.zip")

with zipfile.ZipFile(zip_path, "w") as archive:
    for filename in required_files:
        archive.write(
            DATA_DIR / filename,
            arcname=filename
        )

print("Dataset files verified:")
for filename in required_files:
    print(" -", filename)

print("\nZIP archive created:", zip_path)

# Download to your computer
files.download(str(zip_path))


Dataset files verified:
 - passages.csv
 - queries.csv
 - relevance_audit.csv

ZIP archive created: /content/curriculumrag_dataset.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>